Import necessary packages:

In [1]:
from collections import deque
import itertools
from sage.all import *
from pysat.formula import *
from pysat.solvers import *

# Compute multiplication table for $B(n)$:

We use the matrix representation of $P$ from the Gardam paper (https://arxiv.org/abs/2312.05240), which makes solving the word problem in $P$ reduce to matrix multiplication over integer matrices.

In [2]:
K = GF(2)

# Cyclic group
C = CyclicPermutationGroup(7)

# Dihedral group (on an n-gon)
D = DihedralGroup(3)

# Promislow Group (known units of support length 21)
gens = [matrix(QQ, 4, [1, 0, 0, 1, 0, -1, 0, 1, 0, 0, -1, 0, 0, 0, 0, 1]), matrix(QQ, 4, [-1, 0, 0, 0, 0, 1, 0, 1, 0, 0, -1, 1, 0, 0, 0, 1])]
P = MatrixGroup(gens)

# Wreath Product
G1 = CyclicPermutationGroup(2)
G2 = CyclicPermutationGroup(3)
W = gap.StandardWreathProduct(gap(G1), gap(G2))

# Surface group of genus g (Small cancellation C'(1/7))
# Note: need Knuth-Bendix implementation to get this working
def SurfaceGroup(g):
    """
    Create the surface group of genus g in GAP.
    The surface group has presentation:
    ⟨a₁,b₁,...,aₙ,bₙ | [a₁,b₁]...[aₙ,bₙ] = 1⟩
    where [a,b] = a⁻¹b⁻¹ab is the commutator.
    """
    if g < 0:
        raise ValueError("Genus must be non-negative")
    
    if g == 0:  # Sphere - trivial group
        return gap.TrivialGroup()
    
    # Create free group with 2g generators
    F = FreeGroup(2*g)
    
    # Get the generators
    gens = F.gens()
    
    # Construct the surface relation: [a₁,b₁][a₂,b₂]...[aₙ,bₙ] = 1
    relation = F.one()
    for i in range(1, g):
        a = gens[2*i-1]
        b = gens[2*i]
        commutator = a.inverse() * b.inverse() * a * b
        relation = relation * commutator
    
    # Create the group with the relation
    G = F / [relation,]
    
    return G

# Test if group is from GAP
def is_gap_object(obj):
    from sage.interfaces.gap import GapElement
    return isinstance(obj, GapElement)

# Creates ball of radius n for a GAP group
def create_ball_gap(G, n=5):
    symbols = [gap.One(G),] + list(gap.GeneratorsOfGroup(G)) + [gap.Inverse(A) for A in tuple(gap.GeneratorsOfGroup(G))]
    ball = [prod(word) for word in itertools.product(symbols, repeat=n)]
    ball = list(set(ball))
    return ball

# Creates ball of radius n for any group, GAP or Sage
def create_ball(G, n=5):
    if is_gap_object(G):
        return create_ball_gap(G, n)
    
    symbols = [G.one(),] + list(G.gens()) + [A.inverse() for A in G.gens()]
    ball = [prod(word) for word in itertools.product(symbols, repeat=n)]
    ball = list(set(ball))
    return ball

G = CyclicPermutationGroup(10)
B = create_ball(G, 5)

In [3]:
def create_product_tables(B):
    """
    Creates the product tables given a ball of radius n.

    Input:
    B - the ball of radius n containing group elements

    Output:
    product_table - Python dict where keys are group elements, and values are lists of indices corresponding to factors
    factors_to_product - Python dict where keys are pairs, values are integers. Two keys map to the same integer if their products are the same.
    """
    # Keys are the product, values are lists of pairs realizing the product
    product_table = dict()
    
    factors_to_product = dict() # keys are pairs of indices, values are unique ids for the product. used for the gym environment.
    unique_id = 0
    for i,j in itertools.product(range(len(B)), repeat=2):
        a, b = B[i], B[j]
        val = a*b
        if val in product_table:
            product_table[val].append((i, j))
            factors_to_product[(i,j)] = factors_to_product[product_table[val][0]]
        else:
            product_table[val] = [(i,j),]
            factors_to_product[(i,j)] = unique_id
            unique_id += 1
    return product_table, factors_to_product

product_table, factors_to_product_specific = create_product_tables(B)
print(len(product_table))

10


# Reinforcement Learning Routine

Implements the `ZeroDivisorEnv` as well as the random search, DQN routine, and PPO routine.

In [69]:
import gymnasium as gym
import numpy as np
from gymnasium.spaces import Discrete, MultiBinary
import math

p = 2

class ZeroDivisorEnv(gym.Env):
    def __init__(self, N, factors_to_product):
        self.N = N
        self.factors_to_product = factors_to_product
        self.observation_space = MultiBinary(2*N)
        self.action_space = Discrete(2*N)
        self.alpha = np.zeros(N, dtype=np.int8)
        self.beta = np.zeros(N, dtype=np.int8)

    def get_obs(self):
        return np.concatenate((self.alpha, self.beta))

    def get_info(self):
        return {"support_alpha": np.count_nonzero(self.alpha), "support_beta": np.count_nonzero(self.beta)}

    def reward(self):
        # -1 * support(alpha * beta) / (support(alpha) * support(beta))
        if np.count_nonzero(self.alpha) == 0 or np.count_nonzero(self.beta) == 0:
            return -1 * math.inf
        
        return -1 * self.product_support() / (np.count_nonzero(self.alpha) * np.count_nonzero(self.beta))

    def product_support(self):
        product_dict = dict()
        for i in range(self.N):
            for j in range(self.N):
                if self.alpha[i] and self.beta[j]:
                    val = self.factors_to_product[(i,j)]
                    if val in product_dict:
                        product_dict[val] += 1
                    else:
                        product_dict[val] = 1

        support = 0
        for val in product_dict:
            support += product_dict[val] % p
        return support
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        while np.count_nonzero(self.alpha) <= 0:
            self.alpha = np.random.randint(2, size=self.N, dtype=np.int8)
        while np.count_nonzero(self.beta) <= 0:
            self.beta = np.random.randint(2, size=self.N, dtype=np.int8)
        
        observation = self.get_obs()
        info = self.get_info()

        return observation, info

    def step(self, action):
        if action < self.N:
            self.alpha[action] = (self.alpha[action] + 1) % 2
        elif action >= self.N:
            self.beta[action % self.N] = (self.beta[action % self.N] + 1) % 2

        terminated = True if self.product_support() == 0 else False
        truncated = True if np.count_nonzero(self.alpha) == 0 or np.count_nonzero(self.beta) == 0 else False
        
        reward = self.reward()
        observation = self.get_obs()
        info = self.get_info()

        return observation, reward, terminated, truncated, info
gym.register("ZeroDivisorEnv", ZeroDivisorEnv)

/home/amazin/miniforge3/envs/sage/lib/python3.12/site-packages/gymnasium/envs/registration.py:644: UserWarning: WARN: Overriding environment ZeroDivisorEnv already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [70]:
p = 2
REWARD_THRESHOLD = 0
NUM_ENVS = 8
MAX_EPISODE_STEPS = 1000

def random_search(G, n=5, n_envs=NUM_ENVS, num_steps=MAX_EPISODE_STEPS, seed=None):
    """
    Implements a vectorized pure random search of the ZeroDivisorEnv. The action space is sampled uniformly.
    If an invalid action is taken in any of the environments, the actions are all chosen again.

    Inputs:
    G - the group (can be either a Sage group or a GAP group)
    n - the maximum length of a factor support element to consider
    n_envs - the number of environments to run in parallel
    num_steps - the number of actions to take in a given episode
    seed - the seed for resetting the environment

    Outputs:
    rewards - the vector of the final rewards at the end of the episode
    """
    
    B = create_ball(G, n)
    product_table, factors_to_product = create_product_tables(B)

    ball_size = len(B)
    vec_env = gym.make_vec("ZeroDivisorEnv", num_envs=n_envs, N=ball_size, factors_to_product=factors_to_product)

    obs = vec_env.reset(seed=seed)
    for _ in range(num_steps):
        actions = vec_env.action_space.sample()
        obs, rewards, terminated, truncated, info = vec_env.step(actions)
        while truncated.any():
            obs, rewards, terminated, truncated, info = vec_env.step(actions)
    return rewards

In [71]:
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

In [72]:
TRAINING_STEPS = 25000

def dqn_search(G, n=5, n_envs=NUM_ENVS, num_steps=MAX_EPISODE_STEPS, training_steps=TRAINING_STEPS, dqn_kwargs = None, seed=None):
    """
    Trains a DQN model on the ZeroDivisorEnv.

    Inputs:
    G - the group (can be either a Sage group or a GAP group)
    n - the maximum length of a factor support element to consider
    n_envs - the number of environments to run in parallel
    num_steps - the number of actions to take in a given episode
    training_steps - the number of steps to use for training
    dqn_kwargs (dict) - kwargs to pass to the StableBaselines DQN call
    seed - the seed for resetting the environment

    Outputs:
    rewards - the vector of the final rewards at the end of the episode
    """
    B = create_ball(G, n)
    product_table, factors_to_product = create_product_tables(B)

    ball_size = len(B)
    vec_env = make_vec_env("ZeroDivisorEnv", n_envs=n_envs, env_kwargs={"N": ball_size, "factors_to_product": factors_to_product})
    
    model = DQN("MlpPolicy", vec_env, verbose=0, device="cuda", **dqn_kwargs)
    model.learn(total_timesteps=training_steps)
    model.save("dqn_zerodivisor")

    vec_env.seed(seed)
    obs = vec_env.reset()
    for _ in range(MAX_EPISODE_STEPS):
        action, _states = model.predict(obs)
        obs, rewards, dones, info = vec_env.step(action)
    return rewards

In [73]:
def ppo_search(G, n=5, n_envs=NUM_ENVS, num_steps=MAX_EPISODE_STEPS, training_steps=TRAINING_STEPS):
    """
    Trains a PPO model on the ZeroDivisorEnv.

    Inputs:
    G - the group (can be either a Sage group or a GAP group)
    n - the maximum length of a factor support element to consider
    n_envs - the number of environments to run in parallel
    num_steps - the number of actions to take in a given episode

    Outputs:
    rewards - the vector of the final rewards at the end of the episode
    """
    B = create_ball(G, n)
    product_table, factors_to_product = create_product_tables(B)

    ball_size = len(B)
    vec_env = make_vec_env(ZeroDivisorEnv, n_envs=n_envs, vec_env_cls=SubprocVecEnv, env_kwargs={"N": ball_size, "factors_to_product": factors_to_product})
    
    model = PPO("MlpPolicy", vec_env, verbose=0, device="cpu")
    model.learn(total_timesteps=training_steps)
    model.save("ppo_zerodivisor")
    
    obs = vec_env.reset()
    for _ in range(MAX_EPISODE_STEPS):
        action, _states = model.predict(obs)
        obs, rewards, dones, info = vec_env.step(action)
    return rewards

# RL Experiments

Experiments on various groups with the random and DQN searches.

In [92]:
import pandas as pd

p = 2
REWARD_THRESHOLD = 0
NUM_ENVS = 20
MAX_EPISODE_STEPS = 10_000
TRAINING_STEPS = 50_000

In [96]:
G = DihedralGroup(7)
n = 5
seed = np.random.randint(0, 2 ** 32 - 1)

rand = random_search(G, n, n_envs=NUM_ENVS, seed=seed) # TODO: Make this seed match DQN

dqn_kwargs = dict()
dqn_kwargs['learning_rate'] = 1e-3
dqn_kwargs['batch_size'] = 128
dqn_kwargs['gamma'] = 0.999
dqn_kwargs['exploration_fraction'] = 0.25
dqn_kwargs['train_freq'] = 4
dqn_kwargs['target_update_interval'] = 100_000

dqn = dqn_search(G, n, n_envs=NUM_ENVS, seed=seed, dqn_kwargs=dqn_kwargs) #TODO: Hyperparameter tune this

ppo =[-1 for _ in range(NUM_ENVS)]#  ppo_search(G, n)

percent_diff_dqn = (rand - dqn) / rand * 100
percent_diff_ppo =[-1 for _ in range(NUM_ENVS)] # (rand - ppo) / rand * 100
rewards_dict = {"Random Search": rand, "DQN": dqn, "% Difference DQN": percent_diff_dqn, "PPO": ppo, "% Difference PPO": percent_diff_ppo}

print("Random Search Avg: ", np.average(rand))
print("DQN Avg: ", np.average(dqn))
print("% Diff Avg DQN: ", np.average(percent_diff_dqn))
# print("PPO Avg: ", np.average(ppo))
# print("% Diff Avg PPO: ", np.average(percent_diff_ppo))

df = pd.DataFrame(data=rewards_dict)
print(df)

/home/amazin/miniforge3/envs/sage/lib/python3.12/site-packages/gymnasium/envs/registration.py:736: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes ([]).
  logger.warn(


Random Search Avg:  -0.17761626000262365
DQN Avg:  -0.09152111
% Diff Avg DQN:  19.78895766181605
    Random Search       DQN  % Difference DQN  PPO  % Difference PPO
0       -0.127273 -0.121212          4.761902   -1                -1
1       -0.166667 -0.062500         62.500000   -1                -1
2       -0.125000 -0.150000        -20.000005   -1                -1
3       -0.107143 -0.066667         37.777775   -1                -1
4       -0.142857 -0.093750         34.375000   -1                -1
5       -0.222222 -0.066667         69.999998   -1                -1
6       -0.050000 -0.111111       -122.222224   -1                -1
7       -0.041322 -0.111111       -168.888891   -1                -1
8       -0.095238 -0.072727         23.636366   -1                -1
9       -0.083333 -0.111111        -33.333334   -1                -1
10      -0.333333 -0.111111         66.666666   -1                -1
11      -0.085714 -0.100000        -16.666668   -1                -1
12   

# Assert System of Boolean Equations:

In [ ]:
a_vars = [Atom(f"a_{i}") for i in range(len(B))]
b_vars = [Atom(f"b_{j}") for j in range(len(B))]

x_vars = dict()
cnf = CNF()

# Non triviality
formula = Equals(a_vars[0], PYSAT_TRUE)
cnf.extend([c for c in formula])

formula = Or(*list(a_vars[i] for i in range(1, len(B))))
cnf.extend([c for c in formula])

# Product equations x_g,h = a_g * b_h
for i,j in itertools.product(range(len(B)), repeat=2):
    x_vars[(i, j)] = Atom(f"x_{i}{j}")
    formula = Equals(x_vars[(i,j)], And(a_vars[i], b_vars[j]))
    cnf.extend([c for c in formula])

max_id = -1
for c in cnf.clauses:
    for variable in c:
        if abs(variable) > max_id:
            max_id = abs(variable)

# print(max_id)
next_id = max_id + 1


# Sum equations sum_{gh=k}(x_g,h) = delta(1,k) for each k in the product table
if is_gap_object(G):
    var_list = deque([x_vars[(i,j)] for i,j in product_table[gap.One(G)]])
else:
    var_list = deque([x_vars[(i,j)] for i,j in product_table[G.one()]])
formula_list = []
while var_list:
    if len(var_list) == 1:
        formula = Equals(var_list.pop(), PYSAT_TRUE)
        formula_list.append(formula)
    elif len(var_list) > 2:
        x1 = var_list.pop()
        x2 = var_list.pop()
        aux_var = Atom(next_id)
        var_list.append(aux_var)
        next_id += 1
        formula = Equals(XOr(x1, x2), aux_var)
        formula_list.append(formula)
    elif len(var_list) == 2:
        x1 = var_list.pop()
        x2 = var_list.pop()
        formula = XOr(x1, x2)
        formula_list.append(formula)
for f in formula_list:
    cnf.extend([c for c in f])
        


"""
if len(product_table[P.one()]) > 1:
    formula = XOr(*[x_vars[(i,j)] for i,j in product_table[P.one()]])
    cnf.extend([c for c in formula])
else:
    formula = Equals(x_vars[product_table[P.one()][0]], PYSAT_TRUE)
    cnf.extend([c for c in formula])
"""
for c in cnf.clauses:
    for variable in c:
        if abs(variable) > max_id:
            max_id = abs(variable)
next_id = max_id + 1

formula_list = []
for val in product_table:
    if i
    
    if not val.is_one():
        
        var_list = deque([x_vars[(i,j)] for i,j in product_table[val]])
        while len(var_list) > 0:
            if len(var_list) == 1:
                formula = Neg(var_list.pop())
                formula_list.append(formula)
            elif len(var_list) > 2:
                x1 = var_list.pop()
                x2 = var_list.pop()
                aux_var = Atom(next_id)
                var_list.append(aux_var)
                next_id += 1
                formula = Equals(XOr(x1, x2), aux_var)
                formula_list.append(formula)
            elif len(var_list) == 2:
                x1 = var_list.pop()
                x2 = var_list.pop()
                formula = Neg(XOr(x1, x2))
                formula_list.append(formula)
        
        """
        if len(product_table[val]) > 1:
            formula = Neg(XOr(*[x_vars[(i,j)] for i,j in product_table[val]]))
            cnf.extend([c for c in formula])
        else:
            formula = Neg(x_vars[product_table[val][0]])
            cnf.extend([c for c in formula])
        """
for f in formula_list:
    cnf.extend([c for c in f])

for c in cnf.clauses:
    for variable in c:
        if abs(variable) > max_id:
            max_id = abs(variable)

print(max_id)

In [ ]:
solution = []
has_solution = False
with Minisat22(bootstrap_with=cnf.clauses) as m:
    print(m.solve())
    has_solution = m.solve()
    if has_solution:
        solution = m.get_model()

In [9]:
obj2id = Formula.export_vpool().obj2id

for i in range(len(B)):
    if obj2id[a_vars[i]] in support:
        print(f"a_{i}")
    if obj2id[b_vars[i]] in support:
        print(f"b_{i}")

for i,j in itertools.product(range(len(B)), repeat=2):
    if obj2id[x_vars[(i,j)]] in support:
        print(f"x_{i},{j}")

a_0
b_0
b_2
a_3
b_3
a_4
b_4
a_5
b_5
a_6
a_7
b_7
a_8
b_8
a_9
b_9
a_10
a_13
b_13
a_14
a_15
a_16
a_17
b_17
a_18
a_19
b_19
a_20
b_20
b_22
x_0,0
x_0,2
x_0,3
x_0,4
x_0,5
x_0,7
x_0,8
x_0,9
x_0,13
x_0,17
x_0,19
x_0,20
x_0,22
x_3,0
x_3,2
x_3,3
x_3,4
x_3,5
x_3,7
x_3,8
x_3,9
x_3,13
x_3,17
x_3,19
x_3,20
x_3,22
x_4,0
x_4,2
x_4,3
x_4,4
x_4,5
x_4,7
x_4,8
x_4,9
x_4,13
x_4,17
x_4,19
x_4,20
x_4,22
x_5,0
x_5,2
x_5,3
x_5,4
x_5,5
x_5,7
x_5,8
x_5,9
x_5,13
x_5,17
x_5,19
x_5,20
x_5,22
x_6,0
x_6,2
x_6,3
x_6,4
x_6,5
x_6,7
x_6,8
x_6,9
x_6,13
x_6,17
x_6,19
x_6,20
x_6,22
x_7,0
x_7,2
x_7,3
x_7,4
x_7,5
x_7,7
x_7,8
x_7,9
x_7,13
x_7,17
x_7,19
x_7,20
x_7,22
x_8,0
x_8,2
x_8,3
x_8,4
x_8,5
x_8,7
x_8,8
x_8,9
x_8,13
x_8,17
x_8,19
x_8,20
x_8,22
x_9,0
x_9,2
x_9,3
x_9,4
x_9,5
x_9,7
x_9,8
x_9,9
x_9,13
x_9,17
x_9,19
x_9,20
x_9,22
x_10,0
x_10,2
x_10,3
x_10,4
x_10,5
x_10,7
x_10,8
x_10,9
x_10,13
x_10,17
x_10,19
x_10,20
x_10,22
x_13,0
x_13,2
x_13,3
x_13,4
x_13,5
x_13,7
x_13,8
x_13,9
x_13,13
x_13,17
x_13,19
x_13,20
x_13,22
x_14,0
x_1

# Solve as Multivariate Polynomial System

Uses Singular's `triangular_decomposition`.

In [ ]:
from sage.rings.polynomial.msolve import variety

a_var_names = tuple(f"a_{i}" for i in range(len(B)))
b_var_names = tuple(f"b_{j}" for j in range(len(B)))

R = BooleanPolynomialRing(names=a_var_names+b_var_names)
a_vars = R.gens()[:len(B)]
b_vars = R.gens()[len(B):]

eqns = [sum(a_vars[i]*b_vars[j] for i,j in product_table[P.one()]), a_vars[0] - 1, b_vars[0] - 1]
print("calculating sum eqns...")
for val in product_table:
    if val.is_one():
        continue
    eqns.append(sum(a_vars[i]*b_vars[j] for i,j in product_table[val]))
I = Ideal(eqns)
gb = I.groebner_basis(algorithm='msolve', proof=False)
print(list(gb))
print(I.dimension())
# sorted(variety(I, R, proof=False), key=str)